In [ ]:
!pip -q install pymc arviz holidays

import numpy as np
import pandas as pd

import pymc as pm
import arviz as az

import holidays
from pandas.tseries.offsets import MonthEnd

In [ ]:
monthly_path = "/content/arrivals.csv"
m = pd.read_csv(monthly_path)

# Expect: date like '2010-01' and arrivals numeric
m.columns = [c.strip() for c in m.columns]
if "date" not in m.columns or "arrivals" not in m.columns:
    raise ValueError(f"arrivals.csv must have columns ['date','arrivals']. Found: {m.columns.tolist()}")

m["date"] = pd.to_datetime(m["date"], format="%Y-%m", errors="coerce")
if m["date"].isna().any():
    raise ValueError("Some monthly 'date' values could not be parsed as YYYY-MM.")

m["arrivals"] = pd.to_numeric(m["arrivals"], errors="coerce")
if m["arrivals"].isna().any():
    raise ValueError("Some monthly 'arrivals' values are not numeric.")

m = m.sort_values("date").reset_index(drop=True)
m = m.rename(columns={"date": "month", "arrivals": "Y_month"})

print(m.head())
print(m.tail())
print("Monthly rows:", len(m))

       month  Y_month
0 2010-01-01    50757
1 2010-02-01    57300
2 2010-03-01    52352
3 2010-04-01    38300
4 2010-05-01    35213
         month  Y_month
187 2025-08-01   198235
188 2025-09-01   158971
189 2025-10-01   165193
190 2025-11-01   212906
191 2025-12-01   258928
Monthly rows: 192


In [ ]:
syn_path = "/content/preprocessed-dataset.csv"
syn = pd.read_csv(syn_path)

# Clean column names: strip + replace spaces with underscore
syn.columns = [c.strip().replace(" ", "_") for c in syn.columns]

# You mentioned "preci pitation" (with a space). After cleaning it becomes "preci_pitation".
# We'll keep it as-is.

if "date" not in syn.columns:
    raise ValueError(f"preprocessed-dataset.csv must have a 'date' column. Found: {syn.columns.tolist()}")

syn["date"] = pd.to_datetime(syn["date"], errors="coerce")
if syn["date"].isna().any():
    bad = syn.loc[syn["date"].isna()].head(5)
    raise ValueError(f"Some daily dates couldn't be parsed. Example rows:\n{bad}")

# Rename to ds for merging
syn = syn.rename(columns={"date": "ds"})

# If duplicates per day exist, aggregate numeric cols by mean (safe default)
if syn["ds"].duplicated().any():
    num_cols = syn.select_dtypes(include=[np.number]).columns.tolist()
    syn = syn.groupby("ds", as_index=False)[num_cols].mean()

print("Daily synthetic rows:", len(syn))
print("Columns:", syn.columns.tolist())
syn.head()

FileNotFoundError: [Errno 2] No such file or directory: '/content/preprocessed-dataset.csv'

In [ ]:
start = m["month"].min()
end = (m["month"].max() + MonthEnd(1))  # last day of last month

daily = pd.DataFrame({"ds": pd.date_range(start, end, freq="D")})
daily["month"] = daily["ds"].values.astype("datetime64[M]")  # YYYY-MM-01 month bucket

daily = daily.merge(m, on="month", how="left")

if daily["Y_month"].isna().any():
    missing = daily.loc[daily["Y_month"].isna(), "month"].drop_duplicates()
    raise ValueError(f"Some months missing totals: {missing.tolist()[:10]}")

print(daily.head())
print(daily.tail())
print("Daily calendar rows:", len(daily))

In [ ]:
daily["dow"] = daily["ds"].dt.dayofweek.astype(int)   # 0=Mon..6=Sun
daily["dom"] = daily["ds"].dt.day.astype(int)         # 1..31
daily["is_weekend"] = (daily["dow"] >= 5).astype(int)

# Sri Lanka holidays (change if needed)
lk_holidays = holidays.country_holidays("LK")
daily["is_holiday"] = daily["ds"].dt.date.map(lambda d: int(d in lk_holidays))

In [ ]:
daily = daily.merge(syn, on="ds", how="left")

# Optional: fill gaps if your covariates have missing days
# (common for search series or macro series)
daily = daily.sort_values("ds").reset_index(drop=True)

# Forward fill then back fill numeric covariates
num_cols = daily.select_dtypes(include=[np.number]).columns.tolist()
# We DO NOT fill Y_month; it's already complete
fill_cols = [c for c in num_cols if c not in ["Y_month"]]
daily[fill_cols] = daily[fill_cols].ffill().bfill()

print("Merged daily shape:", daily.shape)
daily.head()

In [ ]:
# Ensure expected columns exist after cleaning
# Your listed columns after cleaning:
# arrivals, gdp_per_capita, inflation_rate, brent_crude_price, usd_lkr, rub_lkr, inr_lkr, gbp_lkr,
# eur_lkr, cny_lkr, web_search, image_search, temperature, preci_pitation, humidity,
# covid_impact_factor, crisis_impact_factor, event_encoded

# --- Proxy column ---
proxy_col = "web_search"
if proxy_col not in daily.columns:
    raise ValueError(f"Proxy column '{proxy_col}' not found. Available: {daily.columns.tolist()}")

proxy = pd.to_numeric(daily[proxy_col], errors="coerce").astype("float64")
proxy = proxy.ffill().bfill().to_numpy()

# --- Feature columns (exclude synthetic 'arrivals'!) ---
feature_cols = [
    # calendar
    "is_weekend", "is_holiday",

    # macro + prices + FX
    "gdp_per_capita", "inflation_rate", "brent_crude_price",
    "usd_lkr", "rub_lkr", "inr_lkr", "gbp_lkr", "eur_lkr", "cny_lkr",

    # search signals (you may include both as features even if web_search is proxy)
    "image_search",

    # weather
    "temperature", "preci_pitation", "humidity",

    # impacts + events
    "covid_impact_factor", "crisis_impact_factor", "event_encoded",
]

# Keep only columns that exist
feature_cols = [c for c in feature_cols if c in daily.columns]

# Build X dataframe
X_df = daily[feature_cols].copy()

# Convert to numeric safely
for c in feature_cols:
    X_df[c] = pd.to_numeric(X_df[c], errors="coerce")

X_df = X_df.fillna(0.0)

# Standardize non-binary features
for c in feature_cols:
    vals = X_df[c].values
    uniq = np.unique(vals)
    is_binary = set(uniq.tolist()).issubset({0.0, 1.0})
    if is_binary:
        continue
    mu = X_df[c].mean()
    sd = X_df[c].std()
    X_df[c] = (X_df[c] - mu) / (sd + 1e-8)

X = X_df.to_numpy(dtype="float64")

print("Feature count:", X.shape[1])
print("Using proxy:", proxy_col)
print("Feature columns:", feature_cols)

In [ ]:
months = pd.Index(daily["month"].unique()).sort_values()
month_groups = [np.where(daily["month"].values == mm)[0] for mm in months]

# Align monthly totals to these months
m_idx = m.set_index("month").loc[months]
Y_month = m_idx["Y_month"].to_numpy().astype("int64")

print("Months:", len(months))
print("Days:", len(daily))
print("Example month total:", months[0], Y_month[0])

In [ ]:
import pytensor.tensor as pt # Add this import at the top of the cell

dow_idx = daily["dow"].to_numpy().astype("int64")           # 0..6
dom_idx = (daily["dom"].to_numpy() - 1).astype("int64")    # 0..30
n_days = len(daily)
n_feat = X.shape[1]

with pm.Model() as model:
    # Feature weights
    beta = pm.Normal("beta", 0.0, 1.0, shape=n_feat)
    x_part = pm.math.dot(X, beta)

    # Day-of-week effect (sum-to-zero)
    dow_raw = pm.Normal("dow_raw", 0.0, 0.5, shape=7)
    dow_eff = dow_raw - pm.math.mean(dow_raw)

    # Smooth day-of-month effect (31) via random walk
    sigma_dom = pm.Exponential("sigma_dom", 2.0)
    dom_rw = pm.GaussianRandomWalk("dom_rw", sigma=sigma_dom, shape=31)
    dom_eff = dom_rw - pm.math.mean(dom_rw)

    intercept = pm.Normal("intercept", 0.0, 1.0)

    # Daily latent score
    eta = intercept + x_part + dow_eff[dow_idx] + dom_eff[dom_idx]

    # Allocate within each month using softmax shares
    # Initialize yhat as a PyTensor variable
    yhat_tensor = pt.zeros(n_days, dtype="float64") # Make it a PyTensor variable
    for mi, day_idx in enumerate(month_groups):
        p_m = pm.math.softmax(eta[day_idx])      # shares within month
        yhat_m = Y_month[mi] * p_m               # allocated daily arrivals (this is a PyTensor variable)
        yhat_tensor = pt.set_subtensor(yhat_tensor[day_idx], yhat_m) # Use pytensor.tensor.set_subtensor

    pm.Deterministic("yhat", yhat_tensor) # Use the constructed PyTensor variable

    # Link to proxy: proxy ~ Normal(alpha + gamma*log(yhat+1), sigma)
    alpha = pm.Normal("alpha", 0.0, 5.0)
    gamma = pm.HalfNormal("gamma", 2.0)
    sigma = pm.Exponential("sigma", 1.0)

    # Use the yhat_tensor for the log calculation
    mu = alpha + gamma * pm.math.log(yhat_tensor + 1.0)
    pm.Normal("proxy_obs", mu=mu, sigma=sigma, observed=proxy)

    idata = pm.sample(
        1000, tune=1000,
        chains=2,
        target_accept=0.9,
        random_seed=42
    )

idata

In [ ]:
y_daily = idata.posterior["yhat"].mean(dim=("chain","draw")).values

daily_out = daily[["ds", "month", "Y_month"]].copy()
daily_out["arrivals_daily_est"] = y_daily

# Verify sums match each month (floating)
check = daily_out.groupby("month").agg(
    monthly_total=("Y_month", "first"),
    allocated=("arrivals_daily_est", "sum")
)
check["abs_error"] = (check["allocated"] - check["monthly_total"]).abs()
print(check.head(12))
print("Max abs monthly sum error:", check["abs_error"].max())

In [ ]:
def integerize_month(group):
    Y = int(group["Y_month"].iloc[0])
    vals = group["arrivals_daily_est"].to_numpy()

    flo = np.floor(vals).astype(int)
    remainder = Y - flo.sum()

    frac = vals - flo
    order = np.argsort(-frac)  # biggest fractional parts first

    if remainder > 0:
        flo[order[:remainder]] += 1
    elif remainder < 0:
        # remove from smallest fractional parts (rare)
        flo[order[remainder:]] -= 1

    out = group.copy()
    out["arrivals_daily_int"] = flo
    return out

daily_int = daily_out.groupby("month", group_keys=False).apply(integerize_month)

# Verify exact sums
check2 = daily_int.groupby("month").agg(
    monthly_total=("Y_month", "first"),
    allocated_int=("arrivals_daily_int", "sum")
)
assert (check2["monthly_total"] == check2["allocated_int"]).all()
print("All monthly totals match exactly ✅")

daily_int.head()

In [ ]:
out_path = "/content/daily_arrivals_disaggregated.csv"
daily_int[["ds", "arrivals_daily_int", "arrivals_daily_est", "month", "Y_month"]].to_csv(out_path, index=False)
print("Saved:", out_path)

In [ ]:
syndf_filtered = syn.drop(columns=['arrivals'])
merged_df = pd.merge(syndf_filtered, daily_int[['ds', 'arrivals_daily_int']], on='ds', how='left')

# Ensure all dates from daily_int are present in the final merged_df
if merged_df['arrivals_daily_int'].isna().any():
    missing_dates = merged_df[merged_df['arrivals_daily_int'].isna()]['ds'].tolist()
    raise ValueError(f"Missing `arrivals_daily_int` for dates: {missing_dates[:10]}")

# Save the updated preprocessed dataset
output_preprocessed_path = "/content/updated_preprocessed_dataset.csv"
merged_df.to_csv(output_preprocessed_path, index=False)

print(f"Updated preprocessed dataset saved to: {output_preprocessed_path}")
print("Columns in updated dataset:", merged_df.columns.tolist())
print("First 5 rows of updated dataset:")
display(merged_df.head())

# Task
Display the `check2` DataFrame to verify that the daily integerized arrivals sum up exactly to the actual monthly totals.

## Display Monthly Sum Verification

### Subtask:
Display the `check2` DataFrame to show that the sum of `arrivals_daily_int` for each month matches the `Y_month` (actual monthly arrivals).


**Reasoning**:
The subtask requires displaying the `check2` DataFrame to verify that the daily integerized arrivals sum up exactly to the actual monthly totals. This DataFrame already contains the necessary columns for visual inspection.



In [ ]:
check2

## Final Task

### Subtask:
Confirm that the daily arrivals sum to the monthly actual values.


## Summary:

### Q&A
The daily integerized arrivals successfully sum up to the actual monthly totals.

### Data Analysis Key Findings
*   The `check2` DataFrame confirmed that the sum of `arrivals_daily_int` for each month, represented by the `allocated_int` column, perfectly matches the `Y_month` (actual monthly arrivals) values in the `monthly_total` column.
*   The values in the `monthly_total` column were identical to the values in the `allocated_int` column for every month.

### Insights or Next Steps
*   The integerization and allocation method used to distribute monthly actuals into daily integerized values effectively preserves the original monthly totals, ensuring data integrity.
*   This successful validation step confirms the reliability of the daily integerized arrival data for further granular analysis or operational planning.
